# Calculate the Annual Energy and Peak Demand by BA

In [1]:
# Start by importing the packages we need:
import os
import datetime
import yaml

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from glob import glob
from matplotlib import pyplot 
from mpl_toolkits.axes_grid1 import make_axes_locatable


## Set the Directory Structure

In [2]:
# Identify the data and impage input and output directories:
ba_to_process_input_directory =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2025_ldrd/data/'
cleaned_ba_data_directory =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2025_ldrd/data/cleaned_historical_data/'


## Set the List of Balancing Authorities to Analyze

BAs used in this analysis are controlled by a master file `balancing_authorities_modeled.yml` stored in the `/data` directory.

In [3]:
# Read the yml file into a dictionary:
with open((ba_to_process_input_directory + 'balancing_authority_modeled.yml'), 'r') as yml:
     ba_list = yaml.load(yml, Loader=yaml.FullLoader)
     bas = [i for i in ba_list.keys()]

# Return the list of BAs to process/plot:
bas


['AZPS', 'BPAT', 'CISO', 'ERCO', 'FPL', 'ISNE', 'PJM', 'SWPP']

## Create a Function to Calculate the Values for a Given Balancing Authority


In [4]:
def calculate_annual_statistics(bas_to_process: list, start_year: int, end_year: int, cleaned_ba_data_directory: str):

    # Initiate a counter and empty dataframe to store the results:
    counter = 0;
    output_df = pd.DataFrame()
    
    # Loop over the BAs and calculate the annual statistics for each one:
    for ba in bas_to_process:
         
        # Read in the cleaned historical dataset:
        ba_df = pd.read_csv((cleaned_ba_data_directory + ba + '_cleaned_historical_data.csv'))

        # Convert the time columns into one datetime variable:
        ba_df['Time_UTC'] = pd.to_datetime(ba_df[['Year', 'Month', 'Day', 'Hour']])
    
        # Convert the populations into millions of people:
        ba_df['Total_Population'] = ba_df['Total_Population']/1000000

        # Only keep the columns that are needed:
        ba_df = ba_df[['Time_UTC', 'Year', 'Total_Population', 'Cleaned_Demand_MWh']].copy()

        # Loop over the years and calculate the statistics for each year:
        for year in range(start_year, end_year):
            # Iterate the counter by one:
            counter = counter + 1

            # Subset the data to just that year:
            year_df = ba_df.loc[(ba_df['Year'] == year)]

            # Calculate the annual statistics and put them into the output dataframe:
            output_df.loc[counter, 'BA'] = ba
            output_df.loc[counter, 'Year'] = int(year)
            output_df.loc[counter, 'Population'] = year_df['Total_Population'].mean().round(2)
            output_df.loc[counter, 'Annual_Energy_TWh'] = ((year_df['Cleaned_Demand_MWh'].sum())*0.000001).round(2)
            output_df.loc[counter, 'Peak_Demand_GW'] = (year_df['Cleaned_Demand_MWh'].max()*0.001).round(2)
    
            # Clean up and move to the next year:
            del year_df

        # Clean up and move to the next BA:
        del ba_df, year
         
    # Set the output file name:
    output_filename = (cleaned_ba_data_directory + 'Annual_Statistics.csv')
   
    # Write out the dataframe to a .csv file:
    output_df.to_csv(output_filename, sep=',', index=False)

    return output_df
    

In [5]:
# Test the function:
output_df = calculate_annual_statistics(bas_to_process = bas,
                                        start_year = 2016,
                                        end_year = 2025,
                                        cleaned_ba_data_directory = cleaned_ba_data_directory)

output_df


,BA,Year,Population,Annual_Energy_TWh,Peak_Demand_GW
1,AZPS,2016.0,6.98,31.36,7.42
2,AZPS,2017.0,7.09,30.91,7.56
3,AZPS,2018.0,7.21,30.28,7.27
4,AZPS,2019.0,7.34,29.64,7.03
5,AZPS,2020.0,7.33,30.84,7.60
...,...,...,...,...,...
68,SWPP,2020.0,18.65,262.06,48.69
69,SWPP,2021.0,18.78,267.49,50.85
70,SWPP,2022.0,18.91,282.51,53.02
71,SWPP,2023.0,18.99,280.58,56.01
